Set up view on crop conditions, crop progress, and find the nass report data closest date to wasde report in aug

In [375]:
from pathlib import Path
import pandas as pd
import numpy as np

Load File Paths

In [376]:
# Project root
BASE_DIR = Path.cwd().parent

# Data files
CC_FILE = BASE_DIR / ".." / "data" / "nass" / "crop_condition.csv"
WASDE_FILE = BASE_DIR / ".." / "data" / "wasde" / "wasde_processed_all_historicals.csv"


In [377]:
df = pd.read_csv(CC_FILE)
df_wasde = pd.read_csv(WASDE_FILE)


WASDE DATA MANIPULATION

In [470]:
selected_month = 8

# Filter once
yield_df = df_wasde[
    (df_wasde["Month"] == selected_month) &
    (df_wasde["crop"] == "nc") &
    (df_wasde["snd_item"] == "yield") &
    (df_wasde["commodity"].isin(["corn", "soybeans", "wheat"]))
].copy()

# Find latest WASDE date for each commodity and year
latest_dates = (
    yield_df
    .groupby(["commodity", "Year"])["Report Date"]
    .transform("max")
)

# Keep only the latest report for each commodity/year
yield_df["release_wasde"] = pd.to_datetime(yield_df["Report Date"])
yield_matrix = yield_df[
    yield_df["Report Date"] == latest_dates
].copy()

#display(yield_matrix)

In [471]:
commodities = ["corn", "soybeans", "wheat"]

matrices = {}

for commodity in commodities:
    matrices[commodity] = (
        yield_matrix[
            yield_matrix["commodity"] == commodity
        ][["Report Date", "Year", "Value","commodity","release_wasde"]]
        .sort_values("Report Date", ascending=True)
        .reset_index(drop=True)
    )
corn_matrix = matrices["corn"]
soy_matrix = matrices["soybeans"]
wheat_matrix = matrices["wheat"]



CROP CONDITIONS DATA MANIPULATIONS

In [472]:
df_cc = df.copy()

df_cc["report_date_cc"] = pd.to_datetime(
    df_cc["Week Ending"],
    errors="coerce"
)

df_cc["Month"] = df_cc["report_date_cc"].dt.month



# Extract condition category
df_cc["Category"] = (
    df_cc["Data Item"]
    .str.extract(r"MEASURED IN PCT (.+)$")[0]
    .str.strip()
)

# Extract crop/type
df_cc["Type"] = (
    df_cc["Data Item"]
    .str.split(" - ", n=1)
    .str[0]
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_cc["Type"].unique()

array(['WHEAT WINTER', 'BARLEY', 'CORN', 'SOYBEANS',
       'WHEAT SPRING (EXCL DURUM)', 'SORGHUM'], dtype=object)

In [473]:
cc_commodities = ['WHEAT WINTER', 'BARLEY', 'CORN', 'SOYBEANS',
       'WHEAT SPRING (EXCL DURUM)', 'SORGHUM']

cc_matrix = df_cc[
    (df_cc["Type"].isin(cc_commodities))&
    (df_cc["Month"] == selected_month)
].copy()

cc_matrix = cc_matrix.pivot_table(
    index=["Type", "report_date_cc"],
    columns="Category",
    values="Value",
    aggfunc="first"
).reset_index()



In [474]:
wheat_to_cc_map = {
    "hard red winter": "WHEAT WINTER",
    "soft red winter": "WHEAT WINTER",
    "hard red spring": "WHEAT SPRING (EXCL DURUM)"
}

grains_to_cc_map = {
    "BARLEY": "barley",
    "CORN": "corn",
    "SOYBEANS": "soybeans",
    "SORGHUM": "sorghum",
}



In [475]:
weights = {
    "EXCELLENT": 110,
    "GOOD": 90,
    "FAIR": 60,
    "POOR": 25,
    "VERY POOR": 5
}

cc_matrix["commodity"] = cc_matrix["Type"].map(grains_to_cc_map)
cc_matrix.sort_values("report_date_cc")
cc_matrix["G+E"] = cc_matrix["GOOD"] +   cc_matrix["EXCELLENT"] 
cc_matrix["condition_index"] = sum(
    cc_matrix[col] * weight
    for col, weight in weights.items()
) / 100

corn_cc = cc_matrix[cc_matrix["Type"] == "CORN"].copy()
soy_cc = cc_matrix[cc_matrix["Type"] == "SOYBEANS"].copy()
winterwheat_cc = cc_matrix[cc_matrix["Type"] == "WHEAT WINTER"].copy()
springwheat_cc = cc_matrix[cc_matrix["Type"] == "WHEAT SPRING (EXCL DURUM)"].copy()


In [476]:
corn_table = pd.merge_asof(
    corn_matrix,
    corn_cc,
    left_on="release_wasde",
    right_on="report_date_cc",
    by="commodity",
    direction="backward",
)

soy_table = pd.merge_asof(
    soy_matrix,
    soy_cc,
    left_on="release_wasde",
    right_on="report_date_cc",
    by="commodity",
    direction="backward",
)

cc_cols = [
    "Year",
    "release_wasde",
    "report_date_cc",
    "EXCELLENT",
    "GOOD",
    "FAIR",
    "POOR",
    "VERY POOR",
    "G+E",
    "condition_index",
    "Value"
]

In [477]:
corn_table[cc_cols]

,Year,release_wasde,report_date_cc,EXCELLENT,GOOD,FAIR,POOR,VERY POOR,G+E,condition_index,Value
0,2010,2010-08-12,2010-08-08,23.0,48.0,19.0,7.0,3.0,71.0,81.80,165.0
1,2011,2011-08-11,2011-08-07,15.0,45.0,24.0,10.0,6.0,60.0,74.20,153.0
2,2012,2012-08-10,2012-08-05,3.0,20.0,27.0,25.0,25.0,23.0,45.00,123.4
3,2013,2013-08-12,2013-08-11,18.0,46.0,25.0,8.0,3.0,64.0,78.35,154.4
4,2014,2014-08-12,2014-08-10,21.0,52.0,20.0,5.0,2.0,73.0,83.25,167.4
5,2015,2015-08-12,2015-08-09,18.0,52.0,21.0,7.0,2.0,70.0,81.05,168.8
6,2016,2016-08-12,2016-08-07,20.0,54.0,19.0,5.0,2.0,74.0,83.35,175.1
7,2017,2017-08-10,2017-08-06,13.0,47.0,27.0,9.0,4.0,60.0,75.25,169.5
8,2018,2018-08-10,2018-08-05,21.0,50.0,19.0,7.0,3.0,71.0,81.40,178.4
9,2019,2019-08-12,2019-08-11,10.0,47.0,30.0,10.0,3.0,57.0,73.95,169.5


In [478]:
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

# 1. Prepare data
reg_df = corn_table[["condition_index", "Value"]].dropna().copy()

# 2. Run linear regression
X = sm.add_constant(reg_df[["condition_index"]])
y = reg_df["Value"]

model = sm.OLS(y, X).fit()

# 3. Scatter plot + regression line
fig = px.scatter(
    reg_df,
    x="condition_index",
    y="Value",
    trendline="ols",
    hover_data=reg_df.columns,
    labels={
        "condition_index": "Corn index (%)",
        "Value": "WASDE corn yield (bu/acre)"
    },
    title="NASS Corn Conditions vs WASDE Yield"
)

fig.update_traces(
    marker=dict(size=9),
    selector=dict(mode="markers")
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

# 4. Regression results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Value   R-squared:                       0.600
Model:                            OLS   Adj. R-squared:                  0.573
Method:                 Least Squares   F-statistic:                     22.50
Date:                Thu, 08 Oct 2026   Prob (F-statistic):           0.000262
Time:                        21:19:13   Log-Likelihood:                -62.059
No. Observations:                  17   AIC:                             128.1
Df Residuals:                      15   BIC:                             129.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              68.7856     21.400     